In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from constants import TRIGGER_GUIDANCE

In [31]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [54]:
CHANNEL_SPECS = {
    "sms_push": {
        "label": "SMS / Uygulama bildirimi",
        "constraint": "En fazla 100 karakter hedefle (160 kesin sınır), tek kısa cümle, yarım bırakma, eksiksiz bir düşünce olsun.",
    },
    "email_casual": {
        "label": "E-posta (orta uzunlukta, profesyonel)",
        "constraint": "2-3 cümle, bir selamlama ve bir kapanış içersin.",
    },
    "email_formal": {
        "label": "E-posta (resmi ton) veya şube araması önerisi",
        "constraint": "2-3 cümle, resmi hitap kullan, gerekirse şube ile görüşmeyi öner.",
    },
}

In [20]:
def choose_channel(age):
    if age < 35:
        return "sms_push"
    elif age < 55:
        return "email_casual"
    else:
        return "email_formal"

In [124]:
def build_prompt(row):
    guidance = TRIGGER_GUIDANCE[row["trigger_type"]]
    channel_key = choose_channel(row["age_at_trigger"])
    channel_spec = CHANNEL_SPECS[channel_key]
    
    context_lines = [
        f"Müşteri yaşı: {row['age_at_trigger']:.0f}",
        f"Cinsiyet: {row['gender']}",
        f"Bölge: {row['district_name']}",
        f"Bölgesel ortalama maaş: {row['avg_salary']:.0f}",
    ]
    if pd.notna(row.get("offer_rate")):
        context_lines.append(f"Kişiye özel aylık faiz oranı: %{row['offer_rate']}")

    system_prompt = (
        "Sen bir bankanın pazarlama içerik asistanısın. Görevin, verilen müşteri bağlamına göre "
        "kısa, kişiselleştirilmiş bir pazarlama mesajı yazmak. Kurallar:\n"
        "- Türkçe yaz.\n"
        "- Mesaj 2-3 cümleyi geçmesin.\n"
        "- Abartılı satış dili kullanma, doğal ve samimi bir ton tut.\n"
        "- SADECE sana verilen bağlamdaki rakamları (faiz oranı, tutar vb.) kullan. "
        "Bağlamda bir oran/rakam verilmemişse, ASLA kendi rakamını uydurma; "
        "bunun yerine 'size özel fırsatlarımızı görüşmek için bizimle iletişime geçin' gibi genel bir ifade kullan.\n"
        "- Her mesaj FARKLI bir açılış cümlesiyle başlasın; kalıplaşmış ifadeleri tekrar tekrar kullanma. "
        "- Müşterinin yaşını, şehrini/bölgesini veya diğer ham profil bilgilerini MESAJIN İÇİNDE ASLA "
        "DOĞRUDAN TEKRARLAMA (örn. '21 yaşında', 'Opava'da yaşayan', 'X şehrindeki' gibi ifadeler YASAK). "
        "Müşterinin yaşı ve bölgesi gibi bağlam bilgilerini doğal görünecek şekilde farklı cümle yapılarında yansıt.\n"
        "Bu bilgileri sadece TON ve YAKLAŞIM seçimini belirlemek için arka planda kullan; "
        "müşteriye yazdığın metinde asla görünür olmasın.\n"
        "- Müşteriye isim vermeden, saygılı bir hitapla yaz.\n"
        "- Banka adı yerine daima tam olarak '[BANKA_ADI]' placeholder'ını kullan.\n"
        "- Her mesajı 'Saygılarımızla, [BANKA_ADI]' ile bitir."
    )

    user_prompt = (
        f"Tetikleyici durum: {guidance['angle']}\n"
        f"Önerilecek ürün: {guidance['product_name']}\n"
        f"Mesaj tonu: {guidance['tone']}\n"
        f"Kanal: {channel_spec['label']}\n"
        f"Kanal kısıtı: {channel_spec['constraint']}\n"
        f"Müşteri bağlamı:\n- " + "\n- ".join(context_lines) + "\n\n"
        "Bu bilgilerle kişiselleştirilmiş bir pazarlama mesajı yaz."
    )

    return system_prompt, user_prompt, channel_key

In [28]:
df = pd.read_csv("data/trigger_table_final.csv")
df.head()

,account_id,year_month,trigger_type,confidence,suggested_product,evidence,gender,age_at_trigger,district_name,avg_salary,unemployment_95,unemployment_96,has_loan,loan_status,loan_at_risk,has_card,card_type,account_age_months,offer_rate
0,1756,1996-09,idle_cash_buildup,1.0,vadeli_mevduat_fon,{'balance_percentile': 1.0},F,41.1,Louny,8965,7.08,8.23,False,NaN,False,True,classic,11,NaN
1,1011,1997-09,idle_cash_buildup,1.0,vadeli_mevduat_fon,{'balance_percentile': 1.0},M,53.3,Jicin,8390,2.28,2.89,True,C,False,False,NaN,33,NaN
2,1011,1997-07,idle_cash_buildup,1.0,vadeli_mevduat_fon,{'balance_percentile': 1.0},M,53.2,Jicin,8390,2.28,2.89,True,C,False,False,NaN,31,NaN
3,1011,1997-06,idle_cash_buildup,1.0,vadeli_mevduat_fon,{'balance_percentile': 1.0},M,53.1,Jicin,8390,2.28,2.89,True,C,False,False,NaN,30,NaN
4,3994,1997-05,idle_cash_buildup,1.0,vadeli_mevduat_fon,{'balance_percentile': 1.0},M,26.6,Cesky Krumlov,9045,3.13,3.60,False,NaN,False,False,NaN,14,NaN


In [46]:
sample_prompt, sample_user_prompt, sample_channel_key = build_prompt(df.iloc[0])

In [47]:
print(sample_prompt)

Sen bir bankanın pazarlama içerik asistanısın. Görevin, verilen müşteri bağlamına göre kısa, kişiselleştirilmiş bir pazarlama mesajı yazmak. Kurallar:
- Türkçe yaz.
- Mesaj 2-3 cümleyi geçmesin.
- Abartılı satış dili kullanma, doğal ve samimi bir ton tut.
- SADECE sana verilen bağlamdaki rakamları (faiz oranı, tutar vb.) kullan. Bağlamda bir oran/rakam verilmemişse, ASLA kendi rakamını uydurma; bunun yerine 'size özel fırsatlarımızı görüşmek için bizimle iletişime geçin' gibi genel bir ifade kullan.
- Müşteriye isim vermeden, saygılı bir hitapla yaz.
- Banka adı yerine daima tam olarak '[BANKA_ADI]' placeholder'ını kullan.
- Her mesajı 'Saygılarımızla, [BANKA_ADI]' ile bitir.


In [48]:
print(sample_user_prompt)

Tetikleyici durum: Hesapta bir süredir birikmiş, değerlendirilmeyen bir bakiye var; bunu değerlendirmesi için nazikçe hatırlat.
Önerilecek ürün: Vadeli Mevduat / Fon
Mesaj tonu: bilgilendirici, fırsat sunan
Kanal: E-posta (orta uzunlukta, profesyonel)
Kanal kısıtı: 2-3 cümle, bir selamlama ve bir kapanış içersin.
Müşteri bağlamı:
- Müşteri yaşı: 41
- Cinsiyet: F
- Bölge: Louny
- Bölgesel ortalama maaş: 8965

Bu bilgilerle kişiselleştirilmiş bir pazarlama mesajı yaz.


In [49]:
print(sample_channel_key)

email_casual


In [29]:
sample = df.groupby("trigger_type").first().reset_index()
print(sample.shape)  # 8 satır bekliyoruz

(8, 19)


In [30]:
sample

,trigger_type,account_id,year_month,confidence,suggested_product,evidence,gender,age_at_trigger,district_name,avg_salary,unemployment_95,unemployment_96,has_loan,loan_status,loan_at_risk,has_card,card_type,account_age_months,offer_rate
0,first_overdraft,2099,1993-12,0.90,ek_limit_kredi_guvence,"{'k_symbol': 'SANKC. UROK', 'event': 'onset'}",M,49.0,Beroun,8980,1.95,2.21,False,B,False,True,classic,11,2.49
1,idle_cash_buildup,1756,1996-09,1.00,vadeli_mevduat_fon,{'balance_percentile': 1.0},F,41.1,Louny,8965,7.08,8.23,False,C,False,True,classic,11,NaN
2,income_increase,1010,1998-01,1.00,yatirim_urunu_limit_artirimi,{'zscore': 11.92},M,57.1,Trebic,8814,4.76,5.74,False,C,False,False,junior,32,NaN
3,liquidity_pressure,3484,1997-04,1.00,esnek_limit_kredi_guvence,{'balance_percentile': 0.0},F,24.0,Domazlice,8620,1.10,1.25,False,A,False,True,classic,34,2.49
4,loan_paid_off,2762,1996-07,0.80,yeni_kredi_teklifi,"{'k_symbol': 'UVER', 'event': 'cessation'}",M,29.4,Praha - zapad,9622,0.45,0.59,True,A,False,False,classic,40,NaN
5,new_recurring_bill,3682,1993-11,0.75,esnek_odeme_bildirim_servisi,"{'k_symbol': 'SIPO', 'event': 'onset'}",M,18.6,Karvina,10177,6.63,7.75,False,A,False,False,gold,5,NaN
6,pension_income_started,1620,1995-07,0.85,emekli_bankaciligi_paketi,"{'k_symbol': 'DUCHOD', 'event': 'onset'}",M,63.9,Pelhrimov,8114,2.38,2.62,False,None,False,False,gold,1,NaN
7,unusual_spending_increase,2179,1995-10,1.00,ihtiyac_kredisi_taksitlendirme,{'zscore': 6.96},M,53.7,Hl.m. Praha,12541,0.29,0.43,False,B,False,False,classic,21,2.39


## ILK LLM CAGRISI

In [50]:
# llm'e eğer channel sms ise -> sms limiti koyuyoruz ki kendi kafasına göre uzatmasın teklif mesajını

def enforce_sms_limit(text, max_chars=160):
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars].rsplit(" ", 1)[0]
    return truncated.rstrip(",;.") + "…"

In [85]:
def generate_message(row, model="gpt-4o-mini"):
    system_prompt, user_prompt, channel_key = build_prompt(row)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.95,
    )
    text = response.choices[0].message.content
    if channel_key == "sms_push":
        text = enforce_sms_limit(text)
    return text, channel_key

In [52]:
for _, row in sample.iterrows():
    message, channel = generate_message(row)
    print(f"\n=== {row['trigger_type']} ({channel}) ===")
    print(message)


=== first_overdraft (email_casual) ===
Sayın Müşterimiz,

Negatif bakiyeye düşmeniz, hayatın getirdiği geçici bir durum olabilir. Bu süreçte, sizin için uygun Ek Limit veya Kredi Güvence ürünlerimizi değerlendirmek isterseniz, size özel %2.49 faiz oranımızla yardımcı olmaktan memnuniyet duyarız. 

Saygılarımızla, [BANKA_ADI]

=== idle_cash_buildup (email_casual) ===
Sayın Müşterimiz,

Bir süredir hesabınızda birikmiş bir bakiye olduğunu fark ettik. Bu bakiyenizi değerlendirmek için [BANKA_ADI] olarak sunduğumuz vadeli mevduat veya fon seçeneklerimizi incelemeyi düşünebilirsiniz. Size özel fırsatlarımızı görüşmek için bizimle iletişime geçin.

Saygılarımızla, [BANKA_ADI]

=== income_increase (email_formal) ===
Değerli Müşterimiz,

Gelirinizdeki artış için sizi tebrik ederiz! Bu fırsatı değerlendirmek adına, yatırım ürünlerimizle geleceğinizi güvence altına alabilir veya limit artırımı ile yeni olanaklar yaratabilirsiniz. Detayları görüşmek için şubemizle iletişime geçmenizi öneririz.



## sample: 50 row'luk sample denemesi (her trigger'a minimum 3 mesaj)

In [55]:
N_TOTAL = 50
MIN_PER_TYPE = 3

trigger_counts = df["trigger_type"].value_counts()
n_types = len(trigger_counts)

n_types

8

In [61]:
# her tipe minimum pay ayır, kalanı orana göre dağıt
base_alloc = {t: MIN_PER_TYPE for t in trigger_counts.index}
remaining = N_TOTAL - (MIN_PER_TYPE * n_types)

base_alloc, remaining

({'idle_cash_buildup': 3,
  'unusual_spending_increase': 3,
  'liquidity_pressure': 3,
  'income_increase': 3,
  'new_recurring_bill': 3,
  'pension_income_started': 3,
  'loan_paid_off': 3,
  'first_overdraft': 3},
 26)

In [59]:
proportions = trigger_counts / trigger_counts.sum()
extra_alloc = (proportions * remaining).round().astype(int)

proportions, extra_alloc

(trigger_type
 idle_cash_buildup            0.366736
 unusual_spending_increase    0.231728
 liquidity_pressure           0.166339
 income_increase              0.162604
 new_recurring_bill           0.051670
 pension_income_started       0.013099
 loan_paid_off                0.004921
 first_overdraft              0.002903
 Name: count, dtype: float64,
 trigger_type
 idle_cash_buildup            10
 unusual_spending_increase     6
 liquidity_pressure            4
 income_increase               4
 new_recurring_bill            1
 pension_income_started        0
 loan_paid_off                 0
 first_overdraft               0
 Name: count, dtype: int64)

In [60]:
final_alloc = {t: base_alloc[t] + extra_alloc.get(t, 0) for t in trigger_counts.index}
print(final_alloc, "toplam:", sum(final_alloc.values()))

{'idle_cash_buildup': np.int64(13), 'unusual_spending_increase': np.int64(9), 'liquidity_pressure': np.int64(7), 'income_increase': np.int64(7), 'new_recurring_bill': np.int64(4), 'pension_income_started': np.int64(3), 'loan_paid_off': np.int64(3), 'first_overdraft': np.int64(3)} toplam: 49


In [ ]:
sample_parts = []
for t, n in final_alloc.items():
    subset = df[df["trigger_type"] == t]
    n = min(n, len(subset))  # o tipte yeterli satır yoksa hepsini al
    sample_parts.append(subset.sample(n=n, random_state=42))

sample_50 = pd.concat(sample_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(sample_50.shape)
sample_50["trigger_type"].value_counts()

(49, 19)


trigger_type
idle_cash_buildup            13
unusual_spending_increase     9
liquidity_pressure            7
income_increase               7
new_recurring_bill            4
loan_paid_off                 3
first_overdraft               3
pension_income_started        3
Name: count, dtype: int64

In [63]:
sample_50.head()

,account_id,year_month,trigger_type,confidence,suggested_product,evidence,gender,age_at_trigger,district_name,avg_salary,unemployment_95,unemployment_96,has_loan,loan_status,loan_at_risk,has_card,card_type,account_age_months,offer_rate
0,1376,1996-03,unusual_spending_increase,0.340207,ihtiyac_kredisi_taksitlendirme,{'zscore': 2.04},F,77.3,Zlin,9624,1.60,2.31,False,NaN,False,False,NaN,6,2.39
1,4995,1997-01,loan_paid_off,0.800000,yeni_kredi_teklifi,"{'k_symbol': 'UVER', 'event': 'cessation'}",M,57.4,Strakonice,8547,2.65,3.64,True,D,True,False,NaN,29,NaN
2,8149,1998-11,first_overdraft,0.900000,ek_limit_kredi_guvence,"{'k_symbol': 'SANKC. UROK', 'event': 'onset'}",F,35.4,Zlin,9624,1.60,2.31,False,NaN,False,False,NaN,18,2.39
3,7184,1998-08,loan_paid_off,0.800000,yeni_kredi_teklifi,"{'k_symbol': 'UVER', 'event': 'cessation'}",F,55.6,Kolin,9307,3.85,4.43,True,A,False,False,NaN,38,NaN
4,3288,1998-06,unusual_spending_increase,0.447204,ihtiyac_kredisi_taksitlendirme,{'zscore': 2.68},F,69.6,Brno - mesto,9897,1.60,1.96,False,NaN,False,False,NaN,45,2.39


In [64]:
import time

results = []
for idx, row in sample_50.iterrows():
    try:
        message, channel_key = generate_message(row)
        results.append({
            "account_id": row["account_id"],
            "year_month": row["year_month"],
            "trigger_type": row["trigger_type"],
            "channel": channel_key,
            "message": message,
        })
    except Exception as e:
        print(f"Hata (satır {idx}, account {row['account_id']}): {e}")
    time.sleep(0.3)  # API rate limit'e takılmamak için küçük bekleme

messages_df = pd.DataFrame(results)
print(messages_df.shape)
messages_df.head(10)

(49, 5)


,account_id,year_month,trigger_type,channel,message
0,1376,1996-03,unusual_spending_increase,email_formal,"Sayın Müşterimiz,\n\nSon harcamalarınızın norm..."
1,4995,1997-01,loan_paid_off,email_formal,"Sayın Müşterimiz,\n\nÖnceki kredinizi başarıyl..."
2,8149,1998-11,first_overdraft,email_casual,"Sayın Müşterimiz,\n\nNegatif bakiyeye düşmeniz..."
3,7184,1998-08,loan_paid_off,email_formal,"Sayın Müşterimiz,\n\nKredinizi başarıyla kapat..."
4,3288,1998-06,unusual_spending_increase,email_formal,"Sayın Müşterimiz,\n\nSon harcamalarınızı incel..."
5,1507,1997-03,liquidity_pressure,email_formal,"Saygıdeğer Müşterimiz,\n\nSon dönemde bakiye d..."
6,3045,1998-02,liquidity_pressure,email_formal,"Değerli Müşterimiz, \n\nSon dönemde bakiye dur..."
7,2960,1997-12,liquidity_pressure,email_casual,"Sayın Müşterimiz,\n\nSon dönemde bakiye durumu..."
8,1723,1998-12,income_increase,sms_push,Gelirinizdeki artış için sizi tebrik ederiz! B...
9,3306,1996-05,unusual_spending_increase,email_casual,"Değerli Müşterimiz,\n\nSon harcamalarınızı inc..."


In [68]:
print(messages_df.loc[0].message)

Sayın Müşterimiz,

Son harcamalarınızın normalin üzerinde olduğunu gözlemledik. İhtiyaç Kredisi veya taksitlendirme seçeneklerimizle size yardımcı olabileceğimizi düşünüyoruz; daha fazla bilgi almak için şubemizi ziyaret etmeyi düşünebilirsiniz. 

Saygılarımızla, [BANKA_ADI]


In [81]:
i = 8

for i in range(i,i+8):
    i +=1
    print(messages_df.loc[i].trigger_type, "\n")
    print(messages_df.loc[i].message,"\n-- \n")

unusual_spending_increase 

Değerli Müşterimiz,

Son harcamalarınızı incelediğimizde, olası ek ihtiyaçlarınızı karşılamak için uygun bir çözüm sunmak istedik. Size özel %2.49 faiz oranıyla ihtiyaç kredisi seçeneklerimiz hakkında daha fazla bilgi almak isterseniz, bizimle iletişime geçmekten çekinmeyin.

Saygılarımızla, [BANKA_ADI] 
-- 

idle_cash_buildup 

Sayın Müşterimiz,

Bir süredir hesabınızda birikmiş bir bakiyeniz bulunuyor. Bu birikimi değerlendirmek için vadeli mevduat veya fon seçeneklerimizi incelemek isterseniz, size özel fırsatlarımızı görüşmek amacıyla şubemizi ziyaret etmenizi öneririz. 

Saygılarımızla, [BANKA_ADI] 
-- 

idle_cash_buildup 

Birikmiş bakiyenizi değerlendirmenin zamanı geldi; vadeli mevduat fırsatlarımızı keşfedin.  
Saygılarımızla, [BANKA_ADI] 
-- 

income_increase 

Gelirinizdeki artış için tebrikler! Yatırım fırsatlarımızı değerlendirmek isterseniz bizimle iletişime geçin.  
Saygılarımızla, [BANKA_ADI] 
-- 

idle_cash_buildup 

Sayın Müşterimiz,

Bir s

In [73]:
for i in range(8, 16):
    print(messages_df.loc[i].message)

Gelirinizdeki artış için sizi tebrik ederiz! Birikimlerinizi değerlendirmek için yatırım ürünlerimizi keşfedin.  
Saygılarımızla, [BANKA_ADI]
Değerli Müşterimiz,

Son harcamalarınızı incelediğimizde, olası ek ihtiyaçlarınızı karşılamak için uygun bir çözüm sunmak istedik. Size özel %2.49 faiz oranıyla ihtiyaç kredisi seçeneklerimiz hakkında daha fazla bilgi almak isterseniz, bizimle iletişime geçmekten çekinmeyin.

Saygılarımızla, [BANKA_ADI]
Sayın Müşterimiz,

Bir süredir hesabınızda birikmiş bir bakiyeniz bulunuyor. Bu birikimi değerlendirmek için vadeli mevduat veya fon seçeneklerimizi incelemek isterseniz, size özel fırsatlarımızı görüşmek amacıyla şubemizi ziyaret etmenizi öneririz. 

Saygılarımızla, [BANKA_ADI]
Birikmiş bakiyenizi değerlendirmenin zamanı geldi; vadeli mevduat fırsatlarımızı keşfedin.  
Saygılarımızla, [BANKA_ADI]
Gelirinizdeki artış için tebrikler! Yatırım fırsatlarımızı değerlendirmek isterseniz bizimle iletişime geçin.  
Saygılarımızla, [BANKA_ADI]
Sayın Müşter

In [82]:
from difflib import SequenceMatcher

idle_msgs = messages_df[messages_df["trigger_type"] == "idle_cash_buildup"]["message"].tolist()

similarities = []
for i in range(len(idle_msgs)):
    for j in range(i + 1, len(idle_msgs)):
        ratio = SequenceMatcher(None, idle_msgs[i], idle_msgs[j]).ratio()
        similarities.append(ratio)

print(f"Ortalama benzerlik: {sum(similarities)/len(similarities):.2f}")
print(f"Maksimum benzerlik: {max(similarities):.2f}")

Ortalama benzerlik: 0.34
Maksimum benzerlik: 0.86


In [83]:
from difflib import SequenceMatcher
import itertools

def avg_similarity(messages):
    if len(messages) < 2:
        return None
    pairs = list(itertools.combinations(messages, 2))
    ratios = [SequenceMatcher(None, a, b).ratio() for a, b in pairs]
    return sum(ratios) / len(ratios), max(ratios)

print(f"{'trigger_type':30s} {'n':>4s} {'avg_sim':>8s} {'max_sim':>8s}")
for t in messages_df["trigger_type"].unique():
    msgs = messages_df[messages_df["trigger_type"] == t]["message"].tolist()
    result = avg_similarity(msgs)
    if result:
        avg_s, max_s = result
        print(f"{t:30s} {len(msgs):>4d} {avg_s:>8.2f} {max_s:>8.2f}")

trigger_type                      n  avg_sim  max_sim
unusual_spending_increase         9     0.42     0.75
loan_paid_off                     3     0.76     0.86
first_overdraft                   3     0.30     0.34
liquidity_pressure                7     0.35     0.57
income_increase                   7     0.36     0.74
idle_cash_buildup                13     0.34     0.86
pension_income_started            3     0.59     0.74
new_recurring_bill                4     0.52     0.81


In [87]:
RETRY_TRIGGERS = ["loan_paid_off", "pension_income_started", "new_recurring_bill"]

# sadece flagged tetikleyicilerin satırlarını yeniden üret
retry_rows = sample_50[sample_50["trigger_type"].isin(RETRY_TRIGGERS)]

new_results = []
for idx, row in retry_rows.iterrows():
    try:
        message, channel_key = generate_message(row)
        new_results.append({
            "account_id": row["account_id"], "year_month": row["year_month"],
            "trigger_type": row["trigger_type"], "channel": channel_key, "message": message,
        })
    except Exception as e:
        print(f"Hata: {e}")
    time.sleep(0.3)

new_df = pd.DataFrame(new_results)

# eski satırları çıkar, yenileri ekle
messages_df = pd.concat([
    messages_df[~messages_df["trigger_type"].isin(RETRY_TRIGGERS)],
    new_df
], ignore_index=True)

print(messages_df.shape)

(49, 5)


In [90]:
new_df

,account_id,year_month,trigger_type,channel,message
0,4995,1997-01,loan_paid_off,email_formal,Başarılı bir şekilde önceki kredinizi kapattığ...
1,7184,1998-08,loan_paid_off,email_formal,Başarınızı kutlamak ve sizi ödüllendirmek isti...
2,2979,1997-12,pension_income_started,email_formal,Yeni emekli olmanız dolayısıyla sizi tebrik ed...
3,1228,1995-04,pension_income_started,email_formal,Emekli maaşınızı almaya başlamanız dolayısıyla...
4,9640,1997-02,loan_paid_off,email_formal,Başarılı bir şekilde kredinizi kapatmanızdan d...
5,1549,1997-04,new_recurring_bill,email_casual,Düzenli ödeme yükümlülüklerinizle ilgili daha ...
6,1309,1995-09,new_recurring_bill,sms_push,Yeni düzenli ödeme yükümlülüğünüzü takip etmek...
7,7894,1996-05,new_recurring_bill,sms_push,Yeni ödeme yükümlülüklerinizi takip etmeniz iç...
8,3548,1993-12,pension_income_started,email_formal,Emekli hayatınızda sizlere destek olmaktan mut...
9,1338,1993-06,new_recurring_bill,email_casual,Yeni düzenli ödeme yükümlülüğünüzü takip etmek...


In [94]:
for idx, row in new_df.iterrows():
    print(row.trigger_type)
    print("\n",row.message,"\n--\n")

loan_paid_off

 Başarılı bir şekilde önceki kredinizi kapattığınız için sizi tebrik ederiz. Yeni kredi fırsatlarımızla, finansal hedeflerinizi daha da ileri taşımak için size yardımcı olabiliriz. Detayları görüşmek için en yakın [BANKA_ADI] şubesine uğrayabilirsiniz.  

Saygılarımızla, [BANKA_ADI] 
--

loan_paid_off

 Başarınızı kutlamak ve sizi ödüllendirmek istiyoruz. Önceki kredisini başarıyla kapatmış olmanız gerçekten takdire şayan. Size özel yeni kredi fırsatlarımızı görüşmek için en yakın [BANKA_ADI] şubesine uğrayabilirsiniz. 

Saygılarımızla, [BANKA_ADI] 
--

pension_income_started

 Yeni emekli olmanız dolayısıyla sizi tebrik ederiz. Size özel avantajlı Emekli Bankacılığı Paketimizi tanımak ve ihtiyaçlarınıza uygun en iyi fırsatları görmek için lütfen şubemizi ziyaret edin ya da bizimle iletişime geçin. 

Saygılarımızla, [BANKA_ADI] 
--

pension_income_started

 Emekli maaşınızı almaya başlamanız dolayısıyla sizi kutlar, bu yeni döneminizde yanınızda olmak isteriz. [BANKA_ADI

In [95]:
for t in RETRY_TRIGGERS:
    msgs = messages_df[messages_df["trigger_type"] == t]["message"].tolist()
    result = avg_similarity(msgs)
    if result:
        avg_s, max_s = result
        print(f"{t:30s} avg={avg_s:.2f} max={max_s:.2f}")

loan_paid_off                  avg=0.31 max=0.48
pension_income_started         avg=0.27 max=0.33
new_recurring_bill             avg=0.45 max=0.83


## EN GUNCEL TRIGGERLAR ICIN MESAJ URETIMI LLM ILE

##EN GUNCEL TRIGGER'LAR ICIN MESAJ URETIMI

In [96]:
latest_trigger_per_account = (
    df.sort_values("year_month")
    .groupby("account_id")
    .tail(1)
    .reset_index(drop=True)
)
print(latest_trigger_per_account.shape)
latest_trigger_per_account["trigger_type"].value_counts()

(4420, 19)


trigger_type
income_increase              1474
idle_cash_buildup            1437
unusual_spending_increase     977
liquidity_pressure            477
loan_paid_off                  33
new_recurring_bill             17
first_overdraft                 5
Name: count, dtype: int64

In [99]:
print("primary_trigger toplam satır:", len(df))
print(df["trigger_type"].value_counts())
print()
print("latest_trigger_per_account toplam satır:", len(latest_trigger_per_account))
print(latest_trigger_per_account["trigger_type"].value_counts())

primary_trigger toplam satır: 56493
trigger_type
idle_cash_buildup            20718
unusual_spending_increase    13091
liquidity_pressure            9397
income_increase               9186
new_recurring_bill            2919
pension_income_started         740
loan_paid_off                  278
first_overdraft                164
Name: count, dtype: int64

latest_trigger_per_account toplam satır: 4420
trigger_type
income_increase              1474
idle_cash_buildup            1437
unusual_spending_increase     977
liquidity_pressure            477
loan_paid_off                  33
new_recurring_bill             17
first_overdraft                 5
Name: count, dtype: int64


## HER TIPTEN EN AZ 15

In [103]:
MIN_PER_TYPE_IN_BATCH = 15

current_counts = latest_trigger_per_account["trigger_type"].value_counts()
all_types = df["trigger_type"].unique()

supplement_parts = []
for t in all_types:
    have = current_counts.get(t, 0)
    need = MIN_PER_TYPE_IN_BATCH - have
    if need > 0:
        already_in = latest_trigger_per_account.loc[
            latest_trigger_per_account["trigger_type"] == t, "account_id"
        ]
        pool = df[
            (df["trigger_type"] == t) &
            (~df["account_id"].isin(already_in))
        ].sort_values("confidence", ascending=False)
        pool = pool.drop_duplicates(subset="account_id", keep="first")
        supplement_parts.append(pool.head(need))

supplement_df = pd.concat(supplement_parts, ignore_index=True) if supplement_parts else pd.DataFrame()
print("Eklenen supplement satır sayısı:", len(supplement_df))
print(supplement_df["trigger_type"].value_counts())

Eklenen supplement satır sayısı: 25
trigger_type
pension_income_started    15
first_overdraft           10
Name: count, dtype: int64


In [104]:
latest_trigger_per_account["is_supplemental"] = False
supplement_df["is_supplemental"] = True

final_batch = pd.concat([latest_trigger_per_account, supplement_df], ignore_index=True)
final_batch = final_batch.drop_duplicates(subset=["account_id", "year_month"])
print(final_batch.shape)
final_batch["trigger_type"].value_counts()

(4445, 20)


trigger_type
income_increase              1474
idle_cash_buildup            1437
unusual_spending_increase     977
liquidity_pressure            477
loan_paid_off                  33
new_recurring_bill             17
first_overdraft                15
pension_income_started         15
Name: count, dtype: int64

## FINAL BATCH URETIMI

In [106]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import os

In [107]:
CHECKPOINT_PATH = "data/messages_checkpoint.csv"
MAX_WORKERS = 8

def safe_generate(row_dict):
    try:
        row = pd.Series(row_dict)
        message, channel_key = generate_message(row)
        return {
            "account_id": row["account_id"], "year_month": row["year_month"],
            "trigger_type": row["trigger_type"], "channel": channel_key,
            "message": message, "is_supplemental": row["is_supplemental"], "error": None,
        }
    except Exception as e:
        return {
            "account_id": row_dict["account_id"], "year_month": row_dict["year_month"],
            "trigger_type": row_dict["trigger_type"], "channel": None,
            "message": None, "is_supplemental": row_dict["is_supplemental"], "error": str(e),
        }

if os.path.exists(CHECKPOINT_PATH):
    done_df = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(zip(done_df["account_id"], done_df["year_month"]))
    print(f"Checkpoint bulundu: {len(done_df)} satır zaten üretilmiş")
else:
    done_df = pd.DataFrame()
    done_keys = set()

remaining = final_batch[
    ~final_batch.apply(lambda r: (r["account_id"], r["year_month"]) in done_keys, axis=1)
]
print(f"Kalan: {len(remaining)} satır")

records = remaining.to_dict(orient="records")
results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(safe_generate, r): r for r in records}
    for i, future in enumerate(tqdm(as_completed(futures), total=len(futures))):
        results.append(future.result())
        if (i + 1) % 200 == 0:
            partial = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            partial.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

final_messages = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
final_messages.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
print(final_messages.shape)

Kalan: 4445 satır


100%|██████████| 4445/4445 [12:07<00:00,  6.11it/s]

(4445, 7)


In [108]:
print("Hata sayısı:", final_messages["error"].notna().sum())
final_messages["trigger_type"].value_counts()

Hata sayısı: 552


trigger_type
income_increase              1474
idle_cash_buildup            1437
unusual_spending_increase     977
liquidity_pressure            477
loan_paid_off                  33
new_recurring_bill             17
first_overdraft                15
pension_income_started         15
Name: count, dtype: int64

In [109]:
error_rows = final_messages[final_messages["error"].notna()]
print(error_rows["error"].value_counts().head(10))

error
Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}    552
Name: count, dtype: int64


In [121]:
from openai import OpenAI

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def generate_message_groq(row, model="openai/gpt-oss-120b"):
    system_prompt, user_prompt, channel_key = build_prompt(row)
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.95,
    )
    text = response.choices[0].message.content
    if channel_key == "sms_push":
        text = enforce_sms_limit(text)
    return text, channel_key

In [125]:
test_rows = final_batch.sample(5, random_state=1)
for _, row in test_rows.iterrows():
    message, channel = generate_message_groq(row)
    print(f"\n=== {row['trigger_type']} ({channel}) ===")
    print(message)


=== liquidity_pressure (sms_push) ===
Esnek limit ve %2.49 faizle finansal esnekliğinizi artırabiliriz.  
Saygılarımızla, [BANKA_ADI]

=== income_increase (email_casual) ===
Merhaba,

Gelirinizdeki artışı bir fırsata dönüştürmek için yatırım ürünlerimizle geleceğinizi güvence altına alabilirsiniz. Size özel seçenekleri görüşmek için bizimle iletişime geçmenizi öneririz.

Saygılarımızla, [BANKA_ADI]

=== liquidity_pressure (email_formal) ===
Değerli müşterimiz, son zamanlarda hesabınızda beklediğiniz hareketliliği yakalayamadığınızı fark ettik. İhtiyacınıza uygun esnek limit ve %2.39 oranındaki kredi güvence seçeneklerimizle yanınızdayız. Detayları şubemizde görüşmek isterseniz, size en uygun çözümü birlikte belirleyebiliriz.  

Saygılarımızla, [BANKA_ADI]

=== idle_cash_buildup (email_formal) ===
Hesabınızda birikmiş olan kullanılmayan bakiyenizin değerlendirilmesi için size birkaç seçenek sunmak isteriz. Vadeli mevduat veya fon ürünlerimiz, bu bakiyenizi güvenli bir şekilde değerlend

## SADECE ERROR ALMIŞ SATIRLARA GROQ ILE MESAJ URETME

In [134]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

MAX_WORKERS = 4  # Groq rate limit'ine karşı temkinli

def safe_generate_groq(row_dict, max_retries=3):
    row = pd.Series(row_dict)
    for attempt in range(max_retries):
        try:
            message, channel_key = generate_message_groq(row)
            return {
                "account_id": row["account_id"], "year_month": row["year_month"],
                "trigger_type": row["trigger_type"], "channel": channel_key,
                "message": message, "is_supplemental": row["is_supplemental"], "error": None,
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((2 ** attempt) + random.random())
                continue
            return {
                "account_id": row_dict["account_id"], "year_month": row_dict["year_month"],
                "trigger_type": row_dict["trigger_type"], "channel": None,
                "message": None, "is_supplemental": row_dict["is_supplemental"], "error": str(e),
            }

In [135]:
# 1) doğru kaynaktan retry_rows oluştur
failed_keys = final_messages.loc[final_messages["error"].notna(), ["account_id", "year_month"]]
retry_rows = final_batch.merge(failed_keys, on=["account_id", "year_month"], how="inner")
print(f"Groq ile denenecek: {len(retry_rows)} satır")
print(retry_rows.columns.tolist())

# 2) üretim döngüsü — retry_rows'u tekrar tanımlayan satır YOK
records = retry_rows.to_dict(orient="records")
groq_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(safe_generate_groq, r): r for r in records}
    for future in tqdm(as_completed(futures), total=len(futures)):
        groq_results.append(future.result())

groq_df = pd.DataFrame(groq_results)
print("Groq ile başarılı:", groq_df["error"].isna().sum())
print("Hâlâ hatalı:", groq_df["error"].notna().sum())

Groq ile denenecek: 552 satır
['account_id', 'year_month', 'trigger_type', 'confidence', 'suggested_product', 'evidence', 'gender', 'age_at_trigger', 'district_name', 'avg_salary', 'unemployment_95', 'unemployment_96', 'has_loan', 'loan_status', 'loan_at_risk', 'has_card', 'card_type', 'account_age_months', 'offer_rate', 'is_supplemental']


100%|██████████| 552/552 [30:19<00:00,  3.30s/it]  

Groq ile başarılı: 184
Hâlâ hatalı: 368


In [136]:
groq_df

,account_id,year_month,trigger_type,channel,message,is_supplemental,error
0,3349,1998-12,unusual_spending_increase,email_casual,"Merhaba, \n\nSon harcamanızda bir ek destek i...",False,None
1,3842,1998-12,idle_cash_buildup,email_formal,Birikiminiz hesabınızda bir süredir değerlendi...,False,None
2,2125,1998-12,unusual_spending_increase,sms_push,Harcamalarınız için %2.39 faizle taksitlendirm...,False,None
3,9640,1998-12,income_increase,email_formal,Gelirinizdeki artışı yeni bir yatırım fırsatın...,False,None
4,181,1998-12,income_increase,sms_push,Gelir artışınızı özel yatırım fırsatımızla değ...,False,None
...,...,...,...,...,...,...,...
547,591,1997-09,pension_income_started,None,None,True,Error code: 429 - {'error': {'message': 'Rate ...
548,1670,1996-01,pension_income_started,None,None,True,Error code: 429 - {'error': {'message': 'Rate ...
549,2549,1993-11,pension_income_started,None,None,True,Error code: 429 - {'error': {'message': 'Rate ...
550,2654,1994-08,pension_income_started,None,None,True,Error code: 429 - {'error': {'message': 'Rate ...


In [137]:
groq_df.iloc[550].error

"Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kpx3gwe5ed5sq7mpdgnah0g6` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199519, Requested 834. Please try again in 2m32.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}"

## DEMO

In [138]:
openai_success = final_messages[final_messages["error"].isna()].copy()
openai_success["provider"] = "OpenAI"
openai_success["model"] = "gpt-4o-mini"

groq_success = groq_df[groq_df["error"].isna()].copy()
groq_success["provider"] = "Groq"
groq_success["model"] = "openai/gpt-oss-120b"

final_messages_complete = pd.concat([openai_success, groq_success], ignore_index=True)
final_messages_clean = final_messages_complete.drop(columns=["error"]).reset_index(drop=True)

print(final_messages_clean.shape)
print(final_messages_clean["provider"].value_counts())

(4077, 8)
provider
OpenAI    3893
Groq       184
Name: count, dtype: int64


In [139]:
demo_ready = final_messages_clean.merge(
    final_batch[["account_id", "year_month", "confidence", "suggested_product", "evidence",
                 "gender", "age_at_trigger", "district_name", "avg_salary", "offer_rate"]],
    on=["account_id", "year_month"],
    how="left"
)
print(demo_ready.shape)
demo_ready[["account_id", "trigger_type", "provider", "model", "channel"]].head(5)

(4077, 16)


,account_id,trigger_type,provider,model,channel
0,459,liquidity_pressure,OpenAI,gpt-4o-mini,sms_push
1,3815,unusual_spending_increase,OpenAI,gpt-4o-mini,email_casual
2,3250,liquidity_pressure,OpenAI,gpt-4o-mini,email_formal
3,713,new_recurring_bill,OpenAI,gpt-4o-mini,email_formal
4,1720,liquidity_pressure,OpenAI,gpt-4o-mini,email_casual


In [140]:
demo_ready.to_csv("data/marketing_messages_full.csv", index=False, encoding="utf-8-sig")
print("Export tamamlandı.")

Export tamamlandı.
